## 3. Random Forest model
In this part, the Random Forest model was implemented. First, using the feature selection criteria created in the previous part of the project, the best feature groups were selected on which the model will be trained. Next, hyperparameter tuning will be performed and the best parameters for each feature group will be selected in the cross-validation process. Finally, models for all groups will be tested on the validation set, and the final model with the best result will be selected and tested on test set.
The measure chosen for evaluating models in the project is *balanced accuracy*. I decided to chose this method because our dataset is unbalanced and this measure gives equal importance to each class, instead of being dominated by the majority class.

In [101]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from ReliefF import ReliefF
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

import pickle

pd.set_option("display.max_columns", 500)
pd.set_option("display.max_rows", 150)

In [4]:
df = pd.read_csv('C:\\Users\\kubas\\OneDrive\\Documents\\ML2\\ProjectClassification\\train_fe.csv')

fr = pd.read_excel('C:\\Users\\kubas\\OneDrive\\Documents\\ML2\\ProjectClassification\\feature_ranking.xlsx', index_col=0)

### Spliting train dataset into train and validation sets

In [5]:
X = df.drop(columns=['claim_status'])
y = df['claim_status']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
df_res = pd.concat([pd.DataFrame(X_train, columns=X_train.columns),
                                pd.Series(y_train, name='claim_status')],
                                axis=1)

In [8]:
df_val = pd.concat([X_val, y_val], axis=1)

In [59]:
y_train.value_counts()

claim_status
0    35085
1     2413
Name: count, dtype: int64

### Choosing features for Random Forest model

#### Benchmark_input
In this approach, we select only those variables for which the Logistic Regression model used for the Elastic Net method had no convergence issues. We then check the correlations between these variables and, based on the general ranking (created in the previous section), we reject the variables with lower rankings.

In [11]:
benchmark_input = fr["EN_coef"].dropna().index.tolist()

In [12]:
benchmark_input

['subscription_length',
 'vehicle_age',
 'customer_age',
 'airbags',
 'displacement',
 'length',
 'width',
 'gross_weight',
 'ncap_rating',
 'is_esc_No',
 'transmission_type_Automatic',
 'transmission_type_Manual',
 'is_front_fog_lights_No',
 'is_rear_window_washer_No',
 'is_brake_assist_No',
 'is_brake_assist_Yes',
 'is_driver_seat_height_adjustable_No',
 'vehicle_age_cut',
 'region_density_sqrt',
 'subscription_length_sqrt']

In [13]:
corr = df_res[benchmark_input].corr("spearman")
foo = pd.DataFrame(corr.abs().unstack().sort_values(ascending=False))
foo = foo[foo[0] != 1]
foo = foo[foo.duplicated() == False]
foo.head(10)

,,0
vehicle_age_cut,vehicle_age,0.999999
is_front_fog_lights_No,is_driver_seat_height_adjustable_No,0.987431
airbags,is_rear_window_washer_No,0.976195
displacement,length,0.954880
width,length,0.946606
is_rear_window_washer_No,is_esc_No,0.944375
is_brake_assist_No,is_driver_seat_height_adjustable_No,0.930175
is_esc_No,airbags,0.924219
is_brake_assist_Yes,is_front_fog_lights_No,0.916957
displacement,ncap_rating,0.909436


We deleate variables with big correlation (based on other criteria such as mi_score or Boruta)

In [14]:
fr.loc[["vehicle_age", "vehicle_age_cut"]]

,sign_fscore,sign_fscore_0_1,corr,mi_score,EN_coef,boruta_rank
vehicle_age,1.370705e-07,1,-0.025009,0.002946,-0.001209,2
vehicle_age_cut,1.538931e-07,1,-0.025002,0.000183,-0.001168,4


In [15]:
benchmark_input.remove("vehicle_age_cut")

In [16]:
fr.loc[["subscription_length", "subscription_length_sqrt"]]

,sign_fscore,sign_fscore_0_1,corr,mi_score,EN_coef,boruta_rank
subscription_length,8.854852e-68,1,0.078742,0.004504,0.011278,1
subscription_length_sqrt,2.098944e-69,1,0.078742,0.005591,0.002676,1


In [17]:
benchmark_input.remove("subscription_length")

In [18]:
fr.loc[["is_front_fog_lights_No", "is_driver_seat_height_adjustable_No"]]

,sign_fscore,sign_fscore_0_1,corr,mi_score,EN_coef,boruta_rank
is_front_fog_lights_No,0.00536,1,-0.012860,0.003997,-0.000252,37
is_driver_seat_height_adjustable_No,0.01091,1,-0.011757,0.004582,-0.000230,33


In [19]:
benchmark_input.remove("is_front_fog_lights_No")

In [20]:
fr.loc[["is_brake_assist_No", "is_brake_assist_Yes"]]

,sign_fscore,sign_fscore_0_1,corr,mi_score,EN_coef,boruta_rank
is_brake_assist_No,0.008545,1,-0.012147,0.005606,-0.000274,64
is_brake_assist_Yes,0.008553,1,0.012147,0.006133,0.000250,30


In [21]:
benchmark_input.remove("is_brake_assist_No")

In [22]:
fr.loc[["transmission_type_Manual", "transmission_type_Automatic"]]

,sign_fscore,sign_fscore_0_1,corr,mi_score,EN_coef,boruta_rank
transmission_type_Manual,NaN,0,-0.000037,0.006983,-0.000271,35
transmission_type_Automatic,0.98414,0,0.000037,0.004747,0.000247,44


In [23]:
benchmark_input.remove("transmission_type_Automatic")

In [24]:
fr.loc[["is_rear_window_washer_No", "airbags"]]

,sign_fscore,sign_fscore_0_1,corr,mi_score,EN_coef,boruta_rank
is_rear_window_washer_No,0.759822,0,-0.001400,0.006391,-0.000214,64
airbags,0.704392,0,0.002366,0.006042,0.000796,28


In [25]:
benchmark_input.remove("is_rear_window_washer_No")

In [26]:
benchmark_input

['vehicle_age',
 'customer_age',
 'airbags',
 'displacement',
 'length',
 'width',
 'gross_weight',
 'ncap_rating',
 'is_esc_No',
 'transmission_type_Manual',
 'is_brake_assist_Yes',
 'is_driver_seat_height_adjustable_No',
 'region_density_sqrt',
 'subscription_length_sqrt']

#### Variance treshold
We select 15 features with the highest variance

In [27]:
var = fr.mi_score.sort_values(ascending=False).index.tolist()[0:15]

In [28]:
var

['is_parking_sensors_Yes',
 'is_esc_No',
 'is_rear_window_wiper_No',
 'is_day_night_rear_view_mirror_No',
 'is_ecw_Yes',
 'is_front_fog_lights_Yes',
 'is_central_locking_Yes',
 'is_tpms_No',
 'is_rear_window_defogger_No',
 'is_parking_camera_No',
 'is_power_door_locks_Yes',
 'is_power_steering_Yes',
 'rear_brakes_type_Drum',
 'is_driver_seat_height_adjustable_Yes',
 'is_speed_alert_Yes']

#### Mutal Information
We select groups of features based on Mutal Information criterium

In [30]:
fr.sort_values("mi_score", ascending=False, inplace=True)

In [31]:
mi_features = fr.iloc[0:10].index.tolist()

In [32]:
mi_features_15 = fr.iloc[0:15].index.tolist()

In [33]:
mi_features_20 = fr.iloc[0:20].index.tolist()

In [34]:
mi_features_25 = fr.iloc[0:25].index.tolist()

#### Boruta
We select features based on Boruta ranking - we select only features for which ranking is in 1-10 

In [35]:
br_features = fr[fr.boruta_rank.isin([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])].index.tolist()

In [36]:
br_features

['subscription_length_sqrt',
 'subscription_length',
 'ncap_rating',
 'vehicle_age',
 'region_density_sqrt',
 'gross_weight',
 'displacement',
 'width',
 'region_density',
 'customer_age',
 'customer_age_log',
 'vehicle_age_cut']

#### Correlation
We select 25 featurse most correlated to target variable

In [37]:
fr["corr_abs"] = np.abs(fr["corr"])
fr.sort_values("corr_abs", ascending=False, inplace=True)
corr_features = fr.iloc[0:25].index.tolist()

#### Permutation Importance
It is a method that measures how much a model’s performance deteriorates when a feature’s values are randomly shuffled. So, if permuting a feature breaks the model, that feature is important, if nothing changes, the feature wasn’t useful

In [38]:
# All features
fe_candidates_all = ['subscription_length', 'vehicle_age', 'customer_age', 'region_density',
       'airbags', 'displacement', 'cylinder', 'turning_radius', 'length',
       'width', 'gross_weight', 'ncap_rating', 'segment_A', 'segment_B1',
       'segment_B2', 'segment_C1', 'segment_C2', 'segment_Utility', 'model_M1',
       'model_M10', 'model_M11', 'model_M2', 'model_M3', 'model_M4',
       'model_M5', 'model_M6', 'model_M7', 'model_M8', 'model_M9',
       'fuel_type_CNG', 'fuel_type_Diesel', 'fuel_type_Petrol',
       'max_torque_113Nm@4400rpm', 'max_torque_170Nm@4000rpm',
       'max_torque_200Nm@1750rpm', 'max_torque_200Nm@3000rpm',
       'max_torque_250Nm@2750rpm', 'max_torque_60Nm@3500rpm',
       'max_torque_82.1Nm@3400rpm', 'max_torque_85Nm@3000rpm',
       'max_torque_91Nm@4250rpm', 'max_power_113.45bhp@4000rpm',
       'max_power_118.36bhp@5500rpm', 'max_power_40.36bhp@6000rpm',
       'max_power_55.92bhp@5300rpm', 'max_power_61.68bhp@6000rpm',
       'max_power_67.06bhp@5500rpm', 'max_power_88.50bhp@6000rpm',
       'max_power_88.77bhp@4000rpm', 'max_power_97.89bhp@3600rpm', 'is_esc_No',
       'is_esc_Yes', 'is_adjustable_steering_No', 'is_adjustable_steering_Yes',
       'is_tpms_No', 'is_tpms_Yes', 'is_parking_sensors_No',
       'is_parking_sensors_Yes', 'is_parking_camera_No',
       'is_parking_camera_Yes', 'rear_brakes_type_Disc',
       'rear_brakes_type_Drum', 'transmission_type_Automatic',
       'transmission_type_Manual', 'steering_type_Electric',
       'steering_type_Manual', 'steering_type_Power', 'is_front_fog_lights_No',
       'is_front_fog_lights_Yes', 'is_rear_window_wiper_No',
       'is_rear_window_wiper_Yes', 'is_rear_window_washer_No',
       'is_rear_window_washer_Yes', 'is_rear_window_defogger_No',
       'is_rear_window_defogger_Yes', 'is_brake_assist_No',
       'is_brake_assist_Yes', 'is_power_door_locks_No',
       'is_power_door_locks_Yes', 'is_central_locking_No',
       'is_central_locking_Yes', 'is_power_steering_No',
       'is_power_steering_Yes', 'is_driver_seat_height_adjustable_No',
       'is_driver_seat_height_adjustable_Yes',
       'is_day_night_rear_view_mirror_No', 'is_day_night_rear_view_mirror_Yes',
       'is_ecw_No', 'is_ecw_Yes', 'is_speed_alert_No', 'is_speed_alert_Yes',
       'vehicle_age_cut', 'customer_age_log', 'region_density_sqrt',
       'subscription_length_sqrt',]

In [39]:
estimator = LogisticRegression(solver="liblinear", max_iter=10000) # We use logistic regression 

selector = RFECV(
    estimator,
    step=1,
    cv=5,
    min_features_to_select=10,
    scoring="balanced_accuracy"  
)

selector.fit(
    df_res.loc[:, rfe_candidates_all].values,
    df_res.loc[:, "claim_status"].values.ravel()
)


selected_features_all = df_res.loc[:, rfe_candidates_all].iloc[:, selector.support_].columns.tolist()

In [40]:
rfe_candidates_all = (
    df_res.loc[:, rfe_candidates_all].iloc[:, selector.support_].columns.tolist()
)

In [46]:
estimator = LogisticRegression(solver="liblinear", max_iter=10000)

estimator.fit(
    df_res.loc[:, rfe_candidates_all].values,
    df_res.loc[:, "claim_status"].values.ravel()
)

result = permutation_importance(
    estimator,
    df_res.loc[:, rfe_candidates_all].values,
    df_res.loc[:, "claim_status"].values.ravel(),
    n_repeats=10,
    scoring="balanced_accuracy",
    random_state=42,
    n_jobs=2,
)

sorted_idx = result.importances_mean.argsort()

In [47]:
sorted_idx = sorted_idx[-25:]

In [48]:
pi_var_prim = df_res.loc[:, rfe_candidates_all].columns[sorted_idx].tolist()

#### Relief
It is an algorithm that scores features by how well they distinguish nearby instances from different classes, while keeping similar instances of the same class close. So, a good feature changes little between close same-class neighbors, but changes a lot between close different-class neighbors

In [49]:
benchmark = ['subscription_length', 'vehicle_age', 'customer_age', 'region_density',
       'airbags', 'displacement', 'cylinder', 'turning_radius', 'length',
       'width', 'gross_weight', 'ncap_rating', 'segment_A', 'segment_B1',
       'segment_B2', 'segment_C1', 'segment_C2', 'segment_Utility', 'model_M1',
       'model_M10', 'model_M11', 'model_M2', 'model_M3', 'model_M4',
       'model_M5', 'model_M6', 'model_M7', 'model_M8', 'model_M9',
       'fuel_type_CNG', 'fuel_type_Diesel', 'fuel_type_Petrol',
       'max_torque_113Nm@4400rpm', 'max_torque_170Nm@4000rpm',
       'max_torque_200Nm@1750rpm', 'max_torque_200Nm@3000rpm',
       'max_torque_250Nm@2750rpm', 'max_torque_60Nm@3500rpm',
       'max_torque_82.1Nm@3400rpm', 'max_torque_85Nm@3000rpm',
       'max_torque_91Nm@4250rpm', 'max_power_113.45bhp@4000rpm',
       'max_power_118.36bhp@5500rpm', 'max_power_40.36bhp@6000rpm',
       'max_power_55.92bhp@5300rpm', 'max_power_61.68bhp@6000rpm',
       'max_power_67.06bhp@5500rpm', 'max_power_88.50bhp@6000rpm',
       'max_power_88.77bhp@4000rpm', 'max_power_97.89bhp@3600rpm', 'is_esc_No',
       'is_esc_Yes', 'is_adjustable_steering_No', 'is_adjustable_steering_Yes',
       'is_tpms_No', 'is_tpms_Yes', 'is_parking_sensors_No',
       'is_parking_sensors_Yes', 'is_parking_camera_No',
       'is_parking_camera_Yes', 'rear_brakes_type_Disc',
       'rear_brakes_type_Drum', 'transmission_type_Automatic',
       'transmission_type_Manual', 'steering_type_Electric',
       'steering_type_Manual', 'steering_type_Power', 'is_front_fog_lights_No',
       'is_front_fog_lights_Yes', 'is_rear_window_wiper_No',
       'is_rear_window_wiper_Yes', 'is_rear_window_washer_No',
       'is_rear_window_washer_Yes', 'is_rear_window_defogger_No',
       'is_rear_window_defogger_Yes', 'is_brake_assist_No',
       'is_brake_assist_Yes', 'is_power_door_locks_No',
       'is_power_door_locks_Yes', 'is_central_locking_No',
       'is_central_locking_Yes', 'is_power_steering_No',
       'is_power_steering_Yes', 'is_driver_seat_height_adjustable_No',
       'is_driver_seat_height_adjustable_Yes',
       'is_day_night_rear_view_mirror_No', 'is_day_night_rear_view_mirror_Yes',
       'is_ecw_No', 'is_ecw_Yes', 'is_speed_alert_No', 'is_speed_alert_Yes',
       'vehicle_age_cut', 'customer_age_log', 'region_density_sqrt',
       'subscription_length_sqrt',]

In [50]:
fs = ReliefF(n_neighbors=25, n_features_to_keep=10)

In [51]:
fs.fit(df_res.loc[:, benchmark].values, df_res.loc[:, "claim_status"].values.ravel())

In [52]:
relief1 = df_res.loc[:, benchmark].iloc[:, fs.top_features[0:10]].columns.tolist()

In [53]:
relief2 = df_res.loc[:, benchmark].iloc[:, fs.top_features[0:15]].columns.tolist()

In [54]:
relief3 = df_res.loc[:, benchmark].iloc[:, fs.top_features[0:20]].columns.tolist()

### Random forrest hyperparameters tuning:
The following function is used to tune hyperparameters in the cross-validation process for the Random Forest model. This function returns the best parameters for a given group of variables (function argument) and displays the balanced accuracy score for each parameter configuration (to better track progress and compare how changing a given parameter impacts the cross validation score).
Hyperparameters we tune:
* **max_samples** - Fraction or number of training samples per tree
* **max_depth** - Maximum depth (number of splits) of each tree
* **min_samples_leaf** - Minimum number of samples required in a leaf node
* **max_features** - Number of features considered at each split
* **n_estimators** - Number of trees in the forest (I decided to not tune this parameter and leave it fixed and equal to 300, to decrease calculation time)

In [71]:
def random_forest_cross_val(
    X_train,
    y_train,
    max_samples_range=None,
    max_depth_range=None,
    min_samples_leaf_range=None,
    max_features_range=None,
    cv=5,
    random_state=42,
    n_jobs=-1,
    n_estimators=300   # This one is fixed to decrease calculation time
):

    if max_samples_range is None:
        max_samples_range = [0.6, 0.8, None]

    if max_depth_range is None:
        max_depth_range = [3, 5, 8, None]

    if min_samples_leaf_range is None:
        min_samples_leaf_range = [1, 5, 10]

    if max_features_range is None:
        max_features_range = ["sqrt", "log2", 0.5]

    best_score = -np.inf
    best_params = None

    for max_depth in max_depth_range:
        for min_leaf in min_samples_leaf_range:
            for max_features in max_features_range:
                for max_samples in max_samples_range:

                    model = RandomForestClassifier(
                        class_weight="balanced", # better for unbalanced datasets
                        n_estimators=n_estimators,
                        criterion="gini", # better for classification problems
                        max_depth=max_depth,
                        min_samples_leaf=min_leaf,
                        max_features=max_features,
                        max_samples=max_samples,
                        random_state=random_state,
                        n_jobs=n_jobs # It decreases calculation time
                    )

                    cv_score = cross_val_score(
                        model,
                        X_train,
                        y_train,
                        cv=cv,
                        scoring="balanced_accuracy",
                        n_jobs=n_jobs
                    ).mean()

                    print(
                        f"max_depth={max_depth}, "
                        f"min_samples_leaf={min_leaf}, "
                        f"max_features={max_features}, "
                        f"max_samples={max_samples} "
                        f"-> balanced_accuracy={cv_score:.4f}"
                    )

                    if cv_score > best_score:
                        best_score = cv_score
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "min_samples_leaf": min_leaf,
                            "max_features": max_features,
                            "max_samples": max_samples
                        }

    print("\nBest parameters found:")
    for k, v in best_params.items():
        print(f"  {k} = {v}")
    print(f"  balanced_accuracy = {best_score:.4f}")

    return best_params

#### Tuning parameters for different feature groups:

##### benchmark (all features in dataset)

In [72]:
best_params = random_forest_cross_val(df_res.loc[:, benchmark].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6075
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.6084
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.6074
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.6009
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.6045
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.6039
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.6077
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.6088
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.6092
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6073
max_depth=3, min_samples_leaf=5, max_fea

##### benchmark_input

In [82]:
best_params = random_forest_cross_val(df_res.loc[:, benchmark_input].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6103
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.6096
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.6101
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.6103
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.6096
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.6101
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.6077
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.6106
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.6099
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6103
max_depth=3, min_samples_leaf=5, max_fea

##### Variance treshold

In [83]:
best_params = random_forest_cross_val(df_res.loc[:, var].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.5082
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.5082
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=5, max_fea

##### Mutal Information

In [84]:
best_params = random_forest_cross_val(df_res.loc[:, mi_features_15].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.5082
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.5082
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.5104
max_depth=3, min_samples_leaf=5, max_fea

In [85]:
best_params = random_forest_cross_val(df_res.loc[:, mi_features_25].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.5767
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.5758
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.5759
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.5775
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.5762
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.5769
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.5769
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.5768
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.5764
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.5772
max_depth=3, min_samples_leaf=5, max_fea

##### Boruta

In [86]:
best_params = random_forest_cross_val(df_res.loc[:, br_features].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6078
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.6079
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.6071
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.6078
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.6079
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.6071
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.6073
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.6075
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.6065
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6078
max_depth=3, min_samples_leaf=5, max_fea

##### Correlation 

In [87]:
best_params = random_forest_cross_val(df_res.loc[:, corr_features].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6097
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.6080
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.6076
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.6113
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.6099
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.6093
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.6069
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.6070
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.6071
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6101
max_depth=3, min_samples_leaf=5, max_fea

##### Permutation importance

In [88]:
best_params = random_forest_cross_val(df_res.loc[:, pi_var_prim].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6045
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.6036
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.6042
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.6045
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.6036
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.6042
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.6080
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.6087
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.6074
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.6041
max_depth=3, min_samples_leaf=5, max_fea

##### Relief

In [89]:
best_params = random_forest_cross_val(df_res.loc[:, relief3].values, df_res.loc[:, "claim_status"].values.ravel())

max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=0.8 -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=1, max_features=sqrt, max_samples=None -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.6 -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=0.8 -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=1, max_features=log2, max_samples=None -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.6 -> balanced_accuracy=0.4917
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=0.8 -> balanced_accuracy=0.4917
max_depth=3, min_samples_leaf=1, max_features=0.5, max_samples=None -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=5, max_features=sqrt, max_samples=0.6 -> balanced_accuracy=0.4896
max_depth=3, min_samples_leaf=5, max_fea

### Now as we chose best hyperparameters for each feature group we can choose best model using validation dataset
Code below calculates calculates balanced accuracy score for each feature group with best parameters for each group. Model with highest score on validation set is then chosen as our final best model for Random Forest algorithm

In [91]:
feature_groups = {
    "benchmark": benchmark,
    "benchmark_input": benchmark_input,
    "var": var,
    "mi_features_15": mi_features_15,
    "mi_features_25": mi_features_25,
    "corr_features": corr_features,
    "pi_var_prim": pi_var_prim,
    "relief3": relief3,
    "br_features": br_features,
}

rf_params = {
    "benchmark": {'max_depth': 5, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_samples': 0.8},
    "benchmark_input": {'max_depth': 5, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_samples': 0.6},
    "var": {'max_depth': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.6},
    "mi_features_15": {'max_depth': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.6},
    "mi_features_25": {'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': None},
    "corr_features": {'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.6},
    "pi_var_prim": {'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.8},
    "relief3": {'max_depth': 3, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_samples': 0.6},
    "br_features": {'max_depth': 5, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_samples': 0.8},
}

best_group = None
best_score = 0
results = {}

for name, features in feature_groups.items():
    params = rf_params.get(name, {})  
    model = RandomForestClassifier(**params,
                                   class_weight = 'balanced',
                                   n_estimators = 300,
                                   criterion = 'gini',
                                   random_state = 42,
                                   n_jobs = -1
                                  )

    scores = cross_val_score(
        model,
        X_train[features],
        y_train,
        cv=5,
        scoring="balanced_accuracy"
    )
    mean_score = scores.mean()
    results[name] = mean_score

    print(f"Feature group: {name}, Balanced Accuracy (CV): {mean_score:.4f}, Params: {params}")

    if mean_score > best_score:
        best_score = mean_score
        best_group = name
        best_model = model


print(f"\nBest group: {best_group} with CV score: {best_score:.4f}")

Feature group: benchmark, Balanced Accuracy (CV): 0.6132, Params: {'max_depth': 5, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'max_samples': 0.8}
Feature group: benchmark_input, Balanced Accuracy (CV): 0.6154, Params: {'max_depth': 5, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_samples': 0.6}
Feature group: var, Balanced Accuracy (CV): 0.5104, Params: {'max_depth': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.6}
Feature group: mi_features_15, Balanced Accuracy (CV): 0.5104, Params: {'max_depth': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.6}
Feature group: mi_features_25, Balanced Accuracy (CV): 0.5793, Params: {'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': None}
Feature group: corr_features, Balanced Accuracy (CV): 0.6138, Params: {'max_depth': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.6}
Feature group: pi_var_prim, Balanced Accuracy (CV): 0.6127, Params: {'max_depth': 5, '

#### Fitting and saving best model
Best feature group: benchmark_input, parameters: {'max_depth': 5, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_samples': 0.6}

In [92]:
best_model = RandomForestClassifier(
                        class_weight="balanced",
                        n_estimators=300,
                        criterion="gini", 
                        max_depth=5,
                        min_samples_leaf=10,
                        max_features=0.5,
                        max_samples=0.6,
                        random_state=42,
                        n_jobs=-1)

In [93]:
best_model.fit(df_res.loc[:, benchmark_input].values, df_res.loc[:, "claim_status"].values.ravel())

,n_estimators,300
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,10
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [94]:
filename = "RF.sav"

In [95]:
pickle.dump(best_model, open(filename, "wb"))

### Testing model on test dataset:

In [96]:
df_test = pd.read_csv('C:\\Users\\kubas\\OneDrive\\Documents\\ML2\\ProjectClassification\\test_fe.csv')

In [98]:
X_test = df_test.loc[:, benchmark_input].values
y_test = df_test.loc[:, "claim_status"].values.ravel()

In [99]:
y_pred = best_model.predict(X_test)

In [100]:
bal_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced accuracy: {bal_acc:.4f}")

Balanced accuracy: 0.6071
